# Creation of a networkX graph based on Train Planning Rules (TPR)

## Import libraries and data

In [64]:
import networkx as nx
import pandas as pd

import xml.etree.ElementTree as ET
import csv

## Inputs

In [65]:
#Change inputs here

nodes_file = "nodes.csv"
Train_Planning_file = "data/TrainPlanningRules.xml"
edges_file = "edges.csv"

## Transformation of the file into a csv file

### Extract nodes

In [66]:
with open(nodes_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "tiploc", "name"])

    for event, elem in ET.iterparse(Train_Planning_file, events=("end",)):
        if elem.tag == "TTPRLocation":
            
            writer.writerow([
                elem.attrib.get("Id"),
                elem.attrib.get("Tiploc"),
                elem.attrib.get("Description")
            ])
            
            elem.clear()

### Extract edges

In [67]:
with open(edges_file, "w", newline="", encoding="utf-8") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(["line_code", "source", "target", "distance_miles"])

    for event, elem in ET.iterparse(Train_Planning_file, events=("end",)):
        if elem.tag == "TTPRLineOfRoute":
            line_code = elem.attrib.get("Abbreviation")
            last_location = None
            last_mileage = None


            for loc_elem in elem.findall(".//TTPRLorLocation"):
                current_location_raw = loc_elem.find("Location").attrib.get("FK")
                #Separation of id and name
                current_location = current_location_raw.split(":")[-1] if ":" in current_location_raw else current_location_raw
                
                try:
                    current_mileage = float(loc_elem.attrib.get("Mileage", 0))
                except ValueError:
                    current_mileage = 0.0

                #If there a previous node, we create the edge
                if last_location is not None:
                    distance = round(abs(current_mileage - last_mileage), 3)
                    writer.writerow([line_code, last_location, current_location, distance])

                last_location = current_location
                last_mileage = current_mileage
            
            elem.clear()

In [68]:
nodes = pd.read_csv(nodes_file)
edges = pd.read_csv(edges_file)

print(f"Nodes number : {len(nodes)}")
print(f"Edges number : {len(edges)}")
print(f"Connected nodes number : {len(set(edges['source']).union(set(edges['target'])))}")

Nodes number : 52713
Edges number : 7488
Connected nodes number : 6764


### Data cleaning

In [69]:
df_nodes = nodes.copy()

#Delete row without ID
df_nodes = df_nodes.dropna(subset=['id'])
df_nodes = df_nodes.drop_duplicates()

#Clean the str and make sure any blank space is there
df_nodes['tiploc'] = df_nodes['tiploc'].astype(str).str.strip()
df_nodes['name'] = df_nodes['name'].astype(str).str.strip()

invalid_values = ["unknown", "none", "nan"]

df_nodes = df_nodes[
    ~df_nodes.drop(columns=["id"])
      .apply(lambda col: col.astype(str).str.strip().str.lower())
      .isin(invalid_values)
      .all(axis=1)
]


df_nodes.to_csv("nodes_clean.csv", index=False)

print("Number of nodes remaining : ", len(df_nodes))

Number of nodes remaining :  12107


In [70]:
df_nodes.head()

,id,tiploc,name
2,3.0,AACHEN,Aachen
3,4.0,ABCWM,Abercwmboi
4,5.0,ABDAPEN,Penywaun
5,6.0,ABDARE,Aberdare
6,7.0,ABDATRE,Trecynon


## Graph creation

In [ ]:
df_nodes['id'] = pd.to_numeric(df_nodes['id'], errors='coerce').fillna(0).astype(int).astype(str)
df_edges['source'] = pd.to_numeric(df_edges['source'], errors='coerce').fillna(0).astype(int).astype(str)
df_edges['target'] = pd.to_numeric(df_edges['target'], errors='coerce').fillna(0).astype(int).astype(str)

#Graph creation
G = nx.from_pandas_edgelist(df_edges, source='source', target='target', edge_attr='distance_miles')

#Quick diagnostic
print("===== Network diagnostic ===== \n")
print(f"Number of edges : {G.number_of_edges()}")
print(f"Number of connected nodes : {G.number_of_nodes()}")

#Connectivity checks

if nx.is_connected(G):
    print("Network is fully connected.")
else:
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    giant_size = len(components[0])
    print(f"Network fragmented in {len(components)} parts.")
    print(f"Giant Component : {giant_size} nodes")
    print(f"Coverage ratio : {(giant_size / nodes_with_edges)*100:.1f}%")
        
#Top 10 most connected nodes
print("\n===== Hubs ===== \n")
degrees = dict(G.degree())
top_hubs = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:13]

for node, deg in top_hubs:

    row = df_nodes[df_nodes['id'] == node]
    if not row.empty:
        name = row['name'].values[0]
        print(f"- {name} (ID: {node}) : {deg} connections")
    else:
        print(f"- ID {node} : {deg} connections (Name not found)")

===== Network diagnostic ===== 

Number of edges : 7289
Number of connected nodes : 6764
Network fragmented in 7 parts.
Giant Component : 6751 nodes
Coverage ratio : 99.8%

===== Hubs ===== 

- Ashford International (ASHFKY) (ID: 360) : 8 connections
- Harlesden Jn (ID: 4120) : 7 connections
- Finsbury Park (ID: 3490) : 6 connections
- Longhedge Jn (ID: 5649) : 6 connections
- Nuneaton (ID: 6624) : 6 connections
- Acton Wells Jn (ID: 102) : 6 connections
- Carlisle South Junction (ID: 11001) : 6 connections
- Regional Boundary (ID: 10984) : 6 connections
- Ely North Jn (ID: 3195) : 6 connections
- Factory Jn (ID: 3337) : 6 connections
- Dundee Central Jn (ID: 2966) : 6 connections
- Portobello Jn (Edinburgh) (ID: 7278) : 6 connections
- Tulse Hill (ID: 9502) : 6 connections
